In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

('cuda:1', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78935)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:4', 'NVIDIA A100-SXM4-80GB', 64395)

# #### Device() ####
# device = cuda:1

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:1)
# edge_attr                (32464, 16)              Tensor (cuda:1)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:1)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [2]:
from modules.train import Loader, LoaderStats
from modules.trainers import ReconstrTrainer, ClassifTrainer
from modules.norm import LogCounts
from modules.loss import NBLoss

from modules.model import CountsAutoencoder
from modules.layers import AttentionSetPooling

In [3]:
from modules.layers import Dims, SetPooling, Sequential
from modules.utils import build_hidden_dims, cloneable, clone_or_init, input_to_dict
import torch
import torch.nn as nn
import torch.nn.functional as F

# typing
from modules.data import GraphDataset
from modules.norm import Normalizer
from torch import Tensor
from torch_geometric.data import Data
from typing import Literal, Union

In [4]:
@cloneable
class NBGLM(nn.Module):
    def __init__(self, dataset:GraphDataset, target_class:int|list[int]|None=None, eps:float=1e-8, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.target_class = target_class
        self.eps = eps

        self.mu = None
        self.theta = None

        # placeholder dims
        self.dims = Dims(dataset=dataset, embed_dim=1, head_dim=None, num_heads=1, method='node')

    def init_with_loader(self, loader:Loader):
        self.num_nodes = loader.num_nodes
        self.num_features = loader.num_features
        self.num_classes = loader.num_classes
        dataloader = loader.train_loader

        _stats = LoaderStats(self.num_nodes, self.num_features, eps=self.eps)

        for batch in dataloader:
            x = _stats.filter_batch(batch, target_class=self.target_class)

            # skip if no samples (after filtering)
            if x is None:
                continue

            # update trackers
            _stats.update(x)

        if _stats.count == 0:
            raise ValueError("No samples found for the specified class(es).")
        
        # compute stats
        mean, _, _ = _stats.compute()

        # global parameters (GLM)
        self.mu = nn.Parameter(mean)
        self.theta = nn.Parameter(loader.stats['theta'])

    def forward(self, input:Union[Data, Tensor, dict], need_weights:bool=False):
        # ensure initialized
        if self.num_nodes is None or self.num_features is None:
            raise ValueError("Normalizer not initialized. Call 'init_with_loader' with a Loader to initialize.")
        
        data = input_to_dict(input)
        x = data['x'].view(-1, self.num_nodes)
        batch_size = x.size(0)
        mu = self.mu.view(1, self.num_nodes).expand(batch_size, -1)

        out = {}
        out['x'] = x
        out['theta'] = self.theta.view(1, self.num_nodes)
        out['batch_size'] = batch_size
        out['num_nodes'] = self.num_nodes
        
        # GLM predictions
        out['x_pred'] = mu.detach()
        out['x_t'] = out['x']
        out['x_t_pred'] = out['x_pred']

        return out


In [5]:
@cloneable
class clGLM(nn.Module):
    def __init__(self, dataset:GraphDataset, target_class:int|list[int]|None=None, eps:float=1e-8, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.target_class = target_class
        self.eps = eps
        self.lin = None

        # placeholder dims
        self.dims = Dims(dataset=dataset, embed_dim=1, head_dim=None, num_heads=1, method='node')

    def init_with_loader(self, loader:Loader):
        self.num_nodes = loader.num_nodes
        self.num_classes = loader.num_classes
        self.lin = nn.Linear(self.num_nodes, self.num_classes)

    def forward(self, input:Union[Data, Tensor, dict], need_weights:bool=False):
        # ensure initialized
        if self.lin is None:
            raise ValueError("Normalizer not initialized. Call 'init_with_loader' with a Loader to initialize.")
        
        data = input_to_dict(input)
        x = data['x'].view(-1, self.num_nodes)
        batch_size = x.size(0)

        # Linear classification
        y_logits: Tensor = self.lin(x)
        out = {}
        out['batch_size'] = batch_size
        out['num_nodes'] = self.num_nodes

        out['y_logits'] = y_logits
        out['y_prob'] = y_logits.softmax(dim=-1)
        out['y_pred'] = y_logits.argmax(dim=-1)

        return out


In [6]:
loader = Loader(
    dataset,
    device=device,
    batch_size=128,
)

nb = NBGLM(dataset)

ae = CountsAutoencoder(
    dataset=dataset,
    embed_dim=128,
    method='node',

    norm_class=LogCounts,
    encoder_class=nn.Linear,
    pooling_class=AttentionSetPooling,
    variational=False,

    hidden_dims=1,
    act_fn=nn.ReLU,
    norm_fn='layer',

    norm_kwargs={
        'libnorm':True,
        'znorm':True,
        'learnable':True
    }
)

trainer = ReconstrTrainer(
    lr=1e-3, 
    trainer_norm_class=LogCounts,
    early_stop=True,
    stop_metric='nb',
    
    kw_keys={'x':'x', 'mu':'x_pred', 'theta':'theta'},
    # kw_keys=('x_t_pred', 'x_t', 'x_pred', 'x', 'theta'),
    metric_keys={'pred':'mu', 'target':'x'},
    
    loss_class=NBLoss,
    # loss_class=MultiLoss,
    # loss_kwargs={
    #     'loss_classes': [nn.MSELoss, NBLoss],
    #     'loss_weights': [0.0, 1.0],
    #     'loss_inputs':[
    #         {'input':'x_t_pred', 'target':'x_t'},
    #         {'x':'x', 'mu':'x_pred', 'theta':'theta'}
    #     ],
    # },

    

)

In [7]:
trainer.run(
    model=nb,
    loader=loader,
    num_epochs=300,
    report_metrics=['loss','kld','nb','mae'],
    verbose=True
)

  0%|          | 0/300 [00:00<?, ?it/s]

Test	 loss=8.1378    nb=8.1504    mae=0.7660



In [8]:
loader = Loader(
    dataset,
    device=device,
    batch_size=128,
)

cl_model = clGLM(dataset)

trainer = ClassifTrainer(
        lr=1e-3, 

        early_stop=True,
        stop_metric='accuracy',
        stop_kwargs={'mode':'max'},

        kw_keys={'input':'y_logits', 'target':'y'},
        metric_keys={'pred':'input', 'target':'target'},
        loss_class=nn.CrossEntropyLoss,
        # loss_kwargs={'label_smoothing':0.1}
)

In [9]:
trainer.run(
    model=cl_model,
    loader=loader,
    num_epochs=300,
    report_metrics=['loss', 'accuracy'],
    verbose=True
)

  0%|          | 0/300 [00:00<?, ?it/s]

Test	 loss=587.3464    accuracy=0.8629



In [10]:
_batch.y

tensor([0, 4, 2, 2, 0, 3, 1, 2, 2, 3, 1, 2, 3, 0, 2, 2, 1, 0, 2, 0, 2, 0, 0, 4,
        3, 0, 4, 2, 4, 2, 3, 2, 0, 3, 0, 2, 2, 2, 0, 2, 1, 2, 2, 2, 2, 2, 2, 4,
        3, 2, 2, 2, 2, 2, 2, 4, 0, 0, 2, 2, 3, 2, 2, 3], device='cuda:1')

In [11]:
trainer.model(_batch)['y_pred']

tensor([0, 4, 2, 2, 0, 3, 1, 2, 2, 1, 1, 2, 3, 0, 2, 2, 1, 0, 2, 0, 2, 0, 0, 4,
        3, 0, 4, 2, 4, 2, 3, 2, 0, 2, 0, 2, 2, 2, 0, 2, 1, 2, 2, 2, 2, 2, 2, 4,
        3, 2, 2, 2, 2, 2, 2, 4, 0, 0, 2, 2, 3, 2, 2, 2], device='cuda:1')